# NVS Benchmark no Google Colab

[Abrir no Colab](https://colab.research.google.com/github/PedroHeinrichSP/TCC-Source-Code/blob/update/notebooks/nvs_benchmark_colab.ipynb) — clique para abrir e rodar o notebook no Google Colab.

Notebook focado em execucao no Colab: clone, setup, benchmark rapido, relatorio HTML, download e backup opcional no Drive.

In [ ]:
# Parametros de execucao
REPO_URL = "https://github.com/PedroHeinrichSP/TCC-Source-Code.git"
REPO_DIR = "/content/TCC"
BRANCH = "update"
RUN_ID = "colab_quick"
USE_GOOGLE_DRIVE = False
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/NVS_Benchmark"

# Resultado real (modo estrito)
STRICT_RESULTS = True
MIN_REQUIRED_PAIRS = 1

In [ ]:
# Clone do repositorio
import os
import shutil
import subprocess
import sys

try:
    __import__("google.colab")
except Exception as exc:
    raise RuntimeError("Este notebook foi desenhado para Google Colab.") from exc

if os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)

subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)
print("Projeto em:", os.getcwd())

In [ ]:
# Setup do ambiente
subprocess.run([sys.executable, "-m", "pip", "install", "--upgrade", "pip"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-e", "."], check=True)
subprocess.run([sys.executable, "-m", "nvs_benchmark.cli", "status"], check=True)

In [ ]:
# Montagem opcional do Google Drive
if USE_GOOGLE_DRIVE:
    drive_mod = __import__("google.colab", fromlist=["drive"])
    drive = getattr(drive_mod, "drive")
    drive.mount("/content/drive")
    print("Drive montado.")
else:
    print("USE_GOOGLE_DRIVE=False, montagem ignorada.")

In [ ]:
# Verificacao de GPU no Colab
import torch
print("CUDA disponivel:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# Download de datasets (catalogo)
subprocess.run([
    sys.executable, "-m", "nvs_benchmark.cli", "install",
    "--catalog-file", "./configs/install_catalog.json",
    "--only", "datasets",
    "--execute"
], check=True)

In [ ]:
# Benchmark rapido (CPU/GPU)
snapshot_file = f"./artifacts/metrics/{RUN_ID}.json"

cmd = [
    sys.executable, "-m", "nvs_benchmark.cli", "method-run",
    "--method", "nerf_static",
    "--dataset", "blender_synthetic",
    "--root", "./data/blender_synthetic/nerf_synthetic/lego",
    "--split", "train",
    "--preset", "quick",
    "--output-dir", "./artifacts",
    "--log-dir", "./logs",
    "--compute-metrics",
    "--snapshot-file", snapshot_file,
    "--append-snapshot",
]

if STRICT_RESULTS:
    cmd.extend(["--strict-results", "--min-required-pairs", str(MIN_REQUIRED_PAIRS)])

subprocess.run(cmd, check=True)

print("Snapshot gerado em:", snapshot_file)

In [ ]:
# Gerar relatorio HTML
report_name = f"{RUN_ID}_report"

cmd = [
    sys.executable, "-m", "nvs_benchmark.cli", "report-generate",
    "--snapshot-file", snapshot_file,
    "--output-dir", "./artifacts/reports",
    "--report-name", report_name,
    "--log-dir", "./logs",
    "--no-pdf",
]

if STRICT_RESULTS:
    cmd.extend(["--strict-snapshot", "--min-methods", "1", "--require-finite-metrics"])

subprocess.run(cmd, check=True)

report_html = f"./artifacts/reports/{report_name}.html"
print("Relatorio HTML:", report_html)

In [ ]:
# Exibir o relatorio no notebook
display_mod = __import__("IPython.display", fromlist=["IFrame"])
IFrame = getattr(display_mod, "IFrame")
IFrame(src=report_html, width=1200, height=700)

In [ ]:
# Compactar e baixar artefatos
import pathlib
colab_files_mod = __import__("google.colab", fromlist=["files"])
files = getattr(colab_files_mod, "files")

zip_path = "/content/nvs_benchmark_artifacts"
archive_file = shutil.make_archive(zip_path, "zip", REPO_DIR, "artifacts")
print("Arquivo:", archive_file)

if pathlib.Path(archive_file).exists():
    files.download(archive_file)

In [ ]:
# Backup opcional dos artefatos no Google Drive
if USE_GOOGLE_DRIVE:
    os.makedirs(DRIVE_OUTPUT_DIR, exist_ok=True)
    metrics_dst = os.path.join(DRIVE_OUTPUT_DIR, "metrics")
    reports_dst = os.path.join(DRIVE_OUTPUT_DIR, "reports")

    if os.path.exists(metrics_dst):
        shutil.rmtree(metrics_dst)
    if os.path.exists(reports_dst):
        shutil.rmtree(reports_dst)

    shutil.copytree("./artifacts/metrics", metrics_dst)
    shutil.copytree("./artifacts/reports", reports_dst)
    print("Backup concluido em:", DRIVE_OUTPUT_DIR)
else:
    print("USE_GOOGLE_DRIVE=False, backup ignorado.")